# 4. Machine learning - random split

This notebook corresponds to the second part of the project, and contains results for the random train/test split. To improve reliability and robustness of the model, a K-fold cross-validation (where K = 5) is used to split the data, and the metrics ($r^2$ and RMSE) from each fold are averaged to give a final overall metric.

The following regression methods are used:
- Linear regression
- Ridge regression
- Random forest regression
- Gradient boosting regression

Each method in turn is fit on the train and test data, and the predictions on the test data are plotted alongside the predictions from the chemical box model (minimal input) and ATom measurements.

There are two sets of features being used:

1. The minimal set of features for the first box model. These are `Temp, Pres, jO3_O2_O1D_CAFS, O3_M, H2O_M, CO_M, CH4_M`
2. These features plus the feature `HO2`. $OH$ and $HO_2$ are linked through interconversion reactions so the purpose of this is to evaluate if predictions are more accurate and hence if model score is improved.

The machine learning predictions will be visualised along with the box model predictions, to save re-calculating $[OH]$ multiple times the box model output is read in as a pre-processed `preprocessed_data_including_box_modelled_oh` file.

A grid search is also performed to identify the optimum hyperparameters from the given search range and if the $r^2$ score improves on the score given by the default hyperparameters.

Because K-fold cross-validation is used involving random splits and averaging, there are no data visualisations for this notebook, and only the metrics are calculated.

---

In [3]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [4]:
df = pd.read_csv("preprocessed_data_including_box_modelled_oh.csv")
print(df.shape)
df.head()

(6550, 12)


,date,Temp,Pres,jO3_O2_O1D_CAFS,O3_M,H2O_M,CO_M,CH4_M,HO2_ATHOS,Altitude,[OH]_measured,[OH]_calc
0,2016-07-29 10:56:10-04:00,249.265,343.334,0.000009,8.640570e-08,0.000447,7.233967e-08,0.000002,6.444,8804.58,2.994316e+06,261295.734530
1,2016-07-29 10:56:40-04:00,248.771,337.478,0.000009,7.601130e-08,0.000332,6.900650e-08,0.000002,5.367,8931.10,2.949089e+06,181180.589632
2,2016-07-29 10:57:10-04:00,248.851,331.741,0.000009,8.291030e-08,0.000158,7.108760e-08,0.000002,4.633,9057.14,2.898024e+06,93251.552514
3,2016-07-29 10:58:40-04:00,246.160,315.391,0.000010,8.132570e-08,0.000169,6.804040e-08,0.000002,6.063,9425.14,2.562488e+06,108485.449742
4,2016-07-29 10:59:10-04:00,245.201,309.998,0.000010,8.148520e-08,0.000207,6.748330e-08,0.000002,6.233,9550.39,2.528521e+06,137061.753143


In [5]:
scaler = StandardScaler()

df_inputs = df[['Temp', 'Pres', 'jO3_O2_O1D_CAFS', 'O3_M', 'H2O_M', 'CO_M', 'CH4_M']]
df_inputs_HO2 = df[['Temp', 'Pres', 'jO3_O2_O1D_CAFS', 'O3_M', 'H2O_M', 'CO_M', 'CH4_M', 'HO2_ATHOS']]

X = df_inputs
X_HO2 = df_inputs_HO2

y = df["[OH]_measured"]

## Part 1: Using the default hyperparameters for each model

### Linear regression
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

LinearRegression fits a linear model with coefficients $w = (w_1, …, w_p)$ where $p$ is the number of features to minimise the residual sum of squares between the observed targets in the dataset, and the targets predicted by the linear approximation.

In [8]:
reg = LinearRegression()
reg_HO2 = LinearRegression()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(reg, X, y, cv=kf, scoring='r2')
print(f"Average r2 score: {round(np.mean(scores), 3)}")

scores = cross_val_score(reg, X_HO2, y, cv=kf, scoring='r2')
print(f"Average r2 score with HO2: {round(np.mean(scores), 3)}")

Average r2 score: 0.48
Average r2 score with HO2: 0.57


In [9]:
scores = cross_val_score(reg, X, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

scores = cross_val_score(reg, X_HO2, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE with HO2: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

Average RMSE: 1.87e+06 molecules cm^-3
Average RMSE with HO2: 1.70e+06 molecules cm^-3


### Ridge regression
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

Ridge regression is an example of a regularisation procedure, which is an extension of the linear model. As well as minimising the mean squared error, it also penalises large coefficient values.

Used when there is collinearity in input values and ordinary least squares would overfit training data.

In [11]:
ridgereg = Ridge(random_state=42)
ridgereg_HO2 = Ridge(random_state=42)

pipeline_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

scores = cross_val_score(pipeline_ridge, X, y, cv=kf, scoring='r2')
print(f"Average r2 score: {round(np.mean(scores), 3)}")

scores = cross_val_score(pipeline_ridge, X_HO2, y, cv=kf, scoring='r2')
print(f"Average r2 score with HO2: {round(np.mean(scores), 3)}")

Average r2 score: 0.48
Average r2 score with HO2: 0.57


In [12]:
scores = cross_val_score(pipeline_ridge, X, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

scores = cross_val_score(pipeline_ridge, X_HO2, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE with HO2: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

Average RMSE: 1.87e+06 molecules cm^-3
Average RMSE with HO2: 1.70e+06 molecules cm^-3


### Random forest
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

Random forest fits a number of decision tree classifiers on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.

In [14]:
rfreg = RandomForestRegressor(random_state=42)
rfreg_HO2 = RandomForestRegressor(random_state=42)

scores = cross_val_score(rfreg, X, y, cv=kf, scoring='r2')
print(f"Average r2 score: {round(np.mean(scores), 3)}")

scores = cross_val_score(rfreg, X_HO2, y, cv=kf, scoring='r2')
print(f"Average r2 score with HO2: {round(np.mean(scores), 3)}")

Average r2 score: 0.715
Average r2 score with HO2: 0.766


In [15]:
scores = cross_val_score(rfreg, X, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

scores = cross_val_score(rfreg, X_HO2, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE with HO2: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

Average RMSE: 1.38e+06 molecules cm^-3
Average RMSE with HO2: 1.25e+06 molecules cm^-3


### Gradient boosting
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html

This algorithm builds an additive model in a forward stage-wise fashion. It produces a predictive model from an ensemble of weak predictive models

In [17]:
pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor())
])

model = GradientBoostingRegressor(random_state=42)

model_HO2 = GradientBoostingRegressor(random_state=42)

scores = cross_val_score(pipeline_gb, X, y, cv=kf, scoring='r2')
print(f"Average r2 score: {round(np.mean(scores), 3)}")

scores = cross_val_score(pipeline_gb, X_HO2, y, cv=kf, scoring='r2')
print(f"Average r2 score with HO2: {round(np.mean(scores), 3)}")

Average r2 score: 0.739
Average r2 score with HO2: 0.75


In [18]:
scores = cross_val_score(pipeline_gb, X, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

scores = cross_val_score(pipeline_gb, X_HO2, y, cv=kf, scoring='neg_mean_squared_error')
print(f"Average RMSE with HO2: {np.sqrt(np.mean(-scores)):.2e} molecules cm^-3")

Average RMSE: 1.32e+06 molecules cm^-3
Average RMSE with HO2: 1.30e+06 molecules cm^-3


### Summary
As can be seen, average model score and RMSE improved significantly, showing that taking samples from each date was a more representative and the algorithm was able to learn from the data better, producing a higher $r^2$ score. However, while the average scores improved, there arose the issue of temporal data leakage. This therefore resulted in cases where the $[OH]$ from e.g. the first data being predicted based on what the model has learned from e.g. the last 2 dates.

## Part 2: Grid search to find the hyperparameters that give the best $r^2$ score
There are no hyperparameters for linear regression, so this method is excluded from the grid search.

### Ridge regression

In [22]:
pipeline_ridge = Pipeline([ # Need data to be normalised
    ('scaler', StandardScaler()),
    ('model', Ridge(random_state=42))
])

In [23]:
param_grid_ridge = {
    'model__alpha': [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]
}

grid_ridge = GridSearchCV(pipeline_ridge, param_grid=param_grid_ridge, cv=kf, scoring='r2').fit(X, y)

print(f"Optimum hyperparameters from grid search: {grid_ridge.best_params_}")
print(f"r2 score with these hyperparameters: {round(grid_ridge.best_score_, 3)}")

Optimum hyperparameters from grid search: {'model__alpha': 1}
r2 score with these hyperparameters: 0.48


### Random forest

In [25]:
rf = RandomForestRegressor(random_state=42)

In [26]:
param_grid_rf = {
    'n_estimators': [10, 50, 100],
    'max_depth': range(1, 10, 1)
}

grid_rf = GridSearchCV(rf, param_grid=param_grid_rf, cv=kf, scoring='r2').fit(X, y)

print(f"Optimum hyperparameters from grid search: {grid_rf.best_params_}")
print(f"r2 score with these hyperparameters: {round(grid_rf.best_score_, 3)}")

Optimum hyperparameters from grid search: {'max_depth': 9, 'n_estimators': 100}
r2 score with these hyperparameters: 0.678


### Gradient boosting

In [28]:
pipeline_gb = Pipeline([ # Need data to be normalised
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(random_state=42))
])

In [29]:
param_grid_gb = {
    'model__n_estimators': [10, 50, 100],
    'model__max_depth': range(1, 10, 1)
}

grid_gb = GridSearchCV(pipeline_gb, param_grid=param_grid_gb, cv=kf, scoring='r2').fit(X,y)

print(f"Optimum hyperparameters from grid search: {grid_gb.best_params_}")
print(f"r2 score with these hyperparameters: {round(grid_gb.best_score_, 3)}")

Optimum hyperparameters from grid search: {'model__max_depth': 8, 'model__n_estimators': 100}
r2 score with these hyperparameters: 0.827


### Summary
Because grid searches can be computationally expensive to run especially for ensemble methods, only a limited search range and hyperparameters was used, which explains the ridge regression score staying constant with the default hyperparameter chosen as the optimum. The random forest score also showed a slight decrease which shows that a higher maximum depth would be required to achieve a higher score. However, the gradient boosting algorithm showed an improvement in $r^2$, suggesting that running with the default of no `max_depth` caused the model to overfit. 
